# Product Recommendation Demo
Notebook này thực hiện demo hệ thống recommendation:
1. Chọn ngẫu nhiên user có lịch sử mua hàng > 2
2. Hiển thị sản phẩm đã mua
3. Dự đoán và recommend 10 sản phẩm tốt nhất

In [ ]:
# Import thư viện cần thiết
import pandas as pd
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from model import CAMRec, collate_fn
from datahelper import AmazonReviewDataset
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. Load dữ liệu và model

In [ ]:
# Load data
base_dir = Path(".")
data_dir = base_dir / "data" / "amazon product"

train_df = pd.read_csv(data_dir / "train.csv")
val_df = pd.read_csv(data_dir / "val.csv")
test_df = pd.read_csv(data_dir / "test.csv")

# Kết hợp tất cả data
df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Fix file paths
df["file_path"] = df["file_path"].apply(lambda x: str(base_dir / x))

print(f"Tổng số records: {len(df)}")
print(f"Số users: {df['reviewerID'].nunique()}")
print(f"Số items: {df['asin'].nunique()}")
print(f"\nCác cột: {df.columns.tolist()}")

In [ ]:
# Tạo user và item mapping
users = {u:i for i,u in enumerate(df['reviewerID'].astype(str).unique())}
items = {a:i for i,a in enumerate(df['asin'].astype(str).unique())}

# Reverse mapping
idx_to_user = {i:u for u,i in users.items()}
idx_to_item = {i:a for a,i in items.items()}

print(f"User mapping: {len(users)} users")
print(f"Item mapping: {len(items)} items")

In [ ]:
# Load trained model
model = CAMRec(n_users=len(users), n_items=len(items), 
               user_dim=128, item_dim=128, proj_dim=256, heads=4).to(device)

# Giả sử model đã được train và save
# model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()
print("Model loaded successfully!")

## 2. Chọn ngẫu nhiên user có lịch sử > 2

In [ ]:
# Đếm số lượng sản phẩm mỗi user đã mua
user_counts = df.groupby('reviewerID').size()
eligible_users = user_counts[user_counts > 2].index.tolist()

print(f"Số users có lịch sử > 2: {len(eligible_users)}")

# Chọn ngẫu nhiên 1 user
selected_user = np.random.choice(eligible_users)
selected_user_idx = users[str(selected_user)]

print(f"\nUser được chọn: {selected_user}")
print(f"User index: {selected_user_idx}")
print(f"Số sản phẩm đã mua: {user_counts[selected_user]}")

## 3. Hiển thị các sản phẩm user đã mua

In [ ]:
# Lấy lịch sử mua hàng của user
user_history = df[df['reviewerID'] == selected_user].copy()
user_history = user_history.sort_values('overall', ascending=False)

print(f"Lịch sử mua hàng của user {selected_user}:")
print(f"{'='*80}")
for idx, row in user_history.iterrows():
    print(f"\nProduct: {row['asin']}")
    print(f"Title: {row.get('title', 'N/A')[:60]}...")
    print(f"Price: ${row.get('price', 0):.2f}")
    print(f"Rating: {row['overall']:.1f}/5.0")
    print(f"Image: {row['file_path']}")

In [ ]:
# Hiển thị hình ảnh các sản phẩm đã mua
n_products = min(6, len(user_history))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (_, row) in enumerate(user_history.head(n_products).iterrows()):
    try:
        img = Image.open(row['file_path']).convert('RGB')
        axes[idx].imshow(img)
        title = row.get('title', 'N/A')[:30]
        axes[idx].set_title(f"{title}...\nRating: {row['overall']:.1f}/5", fontsize=9)
        axes[idx].axis('off')
    except Exception as e:
        axes[idx].text(0.5, 0.5, 'Image not found', ha='center', va='center')
        axes[idx].axis('off')

# Ẩn axes thừa
for idx in range(n_products, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.suptitle(f'Sản phẩm user {selected_user} đã mua', fontsize=14, y=1.02)
plt.show()

## 4. Lọc sản phẩm chưa mua và dự đoán rating

In [ ]:
# Lấy danh sách sản phẩm user đã mua
purchased_items = set(user_history['asin'].astype(str).unique())

# Lọc sản phẩm chưa mua
all_items = df['asin'].astype(str).unique()
unpurchased_items = [item for item in all_items if item not in purchased_items]

print(f"Tổng số sản phẩm: {len(all_items)}")
print(f"Đã mua: {len(purchased_items)}")
print(f"Chưa mua: {len(unpurchased_items)}")

In [ ]:
# Chuẩn bị dữ liệu để dự đoán
from torchvision import transforms

tok = AutoTokenizer.from_pretrained('roberta-base')

img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Tạo dataset cho các sản phẩm chưa mua
predictions = []
errors = []

print("Đang dự đoán rating cho các sản phẩm chưa mua...")
print(f"Sẽ dự đoán cho {min(100, len(unpurchased_items))} sản phẩm đầu tiên...")

with torch.no_grad():
    for idx, item_asin in enumerate(unpurchased_items[:100]):
        try:
            # Lấy thông tin sản phẩm
            item_rows = df[df['asin'].astype(str) == item_asin]
            if len(item_rows) == 0:
                continue
            item_info = item_rows.iloc[0]
            item_idx = items[item_asin]
            
            # Chuẩn bị text input
            text = str(item_info.get('title', ''))
            if not text or text == 'nan':
                text = "No title"
            
            tokens = tok(text, padding='max_length', truncation=True, 
                        max_length=128, return_tensors='pt')
            
            # Chuẩn bị image input
            img_path = item_info['file_path']
            if not Path(img_path).exists():
                errors.append(f"Image not found: {img_path}")
                continue
                
            img = Image.open(img_path).convert('RGB')
            img_tensor = img_transform(img).unsqueeze(0)
            
            # Tạo batch
            batch = {
                'user_idx': torch.tensor([selected_user_idx]).to(device),
                'item_idx': torch.tensor([item_idx]).to(device),
                'input_ids': tokens['input_ids'].to(device),
                'attention_mask': tokens['attention_mask'].to(device),
                'image': img_tensor.to(device),
                'rating': torch.tensor([0.0]).to(device)
            }
            
            # Dự đoán
            pred_rating = model(batch).cpu().item()
            
            predictions.append({
                'asin': item_asin,
                'title': item_info.get('title', 'N/A'),
                'price': item_info.get('price', 0),
                'file_path': item_info['file_path'],
                'predicted_rating': pred_rating
            })
            
            if (idx + 1) % 20 == 0:
                print(f"  Processed {idx + 1}/100 items, {len(predictions)} successful")
                
        except Exception as e:
            errors.append(f"Error processing {item_asin}: {str(e)}")
            continue

print(f"\n✓ Đã dự đoán thành công cho {len(predictions)} sản phẩm")
if errors:
    print(f"⚠ Có {len(errors)} lỗi. Hiển thị 5 lỗi đầu tiên:")
    for err in errors[:5]:
        print(f"  - {err}")

## 5. Recommend top 10 sản phẩm

In [ ]:
# Sắp xếp theo predicted rating
recommendations = sorted(predictions, key=lambda x: x['predicted_rating'], reverse=True)[:10]

print(f"\nTOP 10 SẢN PHẨM ĐƯỢC RECOMMEND CHO USER {selected_user}")
print(f"{'='*80}\n")

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. Product ID: {rec['asin']}")
    print(f"   Title: {rec['title'][:60]}...")
    print(f"   Price: ${rec['price']:.2f}")
    print(f"   Predicted Rating: {rec['predicted_rating']:.2f}/5.0")
    print()

In [ ]:
# Hiển thị hình ảnh top 10 sản phẩm recommend
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for idx, rec in enumerate(recommendations):
    try:
        img = Image.open(rec['file_path']).convert('RGB')
        axes[idx].imshow(img)
        title = rec['title'][:25]
        axes[idx].set_title(f"#{idx+1}: {title}...\n"
                           f"Pred: {rec['predicted_rating']:.2f}/5.0\n"
                           f"${rec['price']:.2f}", 
                           fontsize=9)
        axes[idx].axis('off')
    except Exception as e:
        axes[idx].text(0.5, 0.5, 'Image not found', ha='center', va='center')
        axes[idx].axis('off')

plt.tight_layout()
plt.suptitle(f'TOP 10 Sản phẩm được recommend cho user {selected_user}', 
             fontsize=16, y=1.02)
plt.show()